In [ ]:
import sys
!{sys.executable} -m pip install nbformat>=4.2.0 ipywidgets

# EXP_009bFIX: The Lucier Resonance — "The Grain" (Fixed)

## Scientific Objective (Plain Language)

**What are we doing?**
We isolate each of the 144 individual attention heads (12 layers × 12 heads) and loop a signal through each one's OV matrix (`W_V @ W_O`) repeatedly. Each head is a tiny 'room' with its own geometric shape.

**What will we see?**
- A 12×12 heatmap showing which heads are the strongest resonant filters
- A 'voice map' showing what token each head naturally gravitates toward
- Convergence curves for the most interesting heads

**Fixes applied from EXP_009b:**
- `ln_final` applied in `get_top_tokens` for correct token decoding
- Token sanitization in voice map (no broken newlines in Markdown)
- Extended schedule to 500 iterations
- `nbformat` auto-install for Plotly rendering

---

In [ ]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")
print(f"Architecture: {model.cfg.n_layers}L × {model.cfg.n_heads}H, d_head={model.cfg.d_head}")

In [ ]:
# ============================================================
# STEP 2: CONFIGURATION
# ============================================================

ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100, 250, 500]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

PROBE_PROMPT = "Am I sitting in a room different from the one you are in now"

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Probe prompt: '{PROBE_PROMPT}'")
print(f"Heads to scan: {model.cfg.n_layers} × {model.cfg.n_heads} = {model.cfg.n_layers * model.cfg.n_heads}")

In [ ]:
# ============================================================
# STEP 3: EXTRACT OV MATRICES
# ============================================================

def get_ov_matrix(model, layer, head):
    """Extract W_OV = W_V @ W_O for a specific head.
    This is the linear map (d_model -> d_model) that defines
    what the head 'does' to the residual stream."""
    W_V = model.W_V[layer, head]  # (d_model, d_head)
    W_O = model.W_O[layer, head]  # (d_head, d_model)
    return W_V @ W_O              # (d_model, d_model)

test_ov = get_ov_matrix(model, 0, 0)
print(f"OV matrix shape: {test_ov.shape}")
print(f"OV matrix norm: {test_ov.norm().item():.4f}")

In [ ]:
# ============================================================
# STEP 4: THE GRAIN LOOP — Single-Head Resonance (FIXED)
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions.
    FIX: Applies Final LayerNorm before unembedding."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_head_resonance(model, prompt, layer, head, max_iter, schedule):
    """
    Loop a signal through a single head's OV circuit.
    This is power iteration on W_OV.
    """
    W_OV = get_ov_matrix(model, layer, head)
    
    # Get initial activation from forward pass
    hook_point = f"blocks.{layer}.hook_resid_pre"
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point
        )
    current_vector = cache[hook_point][0, -1, :].clone()
    
    snapshots = []
    
    if 0 in schedule:
        snapshots.append({
            "iteration": 0,
            "vector": current_vector.clone().cpu(),
            "norm": current_vector.norm().item(),
            "cosine_sim_to_prev": 1.0,
            "top_tokens": get_top_tokens(model, current_vector),
        })
    
    prev_vector = current_vector.clone()
    
    for i in range(1, max_iter + 1):
        # Apply the head's OV matrix
        current_vector = current_vector @ W_OV
        
        # Normalize to prevent explosion/collapse
        current_norm = current_vector.norm()
        if current_norm > 0:
            current_vector = current_vector / current_norm * prev_vector.norm()
        
        if i in schedule:
            cos_sim = torch.nn.functional.cosine_similarity(
                current_vector.unsqueeze(0),
                prev_vector.unsqueeze(0)
            ).item()
            snapshots.append({
                "iteration": i,
                "vector": current_vector.clone().cpu(),
                "norm": current_norm.item(),
                "cosine_sim_to_prev": cos_sim,
                "top_tokens": get_top_tokens(model, current_vector),
            })
        
        prev_vector = current_vector.clone()
    
    return snapshots

print("Grain resonance engine loaded (FIXED).")

In [ ]:
# ============================================================
# STEP 5: FULL HEAD SWEEP
# ============================================================

head_results = {}

total_heads = model.cfg.n_layers * model.cfg.n_heads
pbar = tqdm(total=total_heads, desc="Scanning heads")

for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        snapshots = run_head_resonance(
            model, PROBE_PROMPT,
            layer, head,
            MAX_ITERATIONS, ITERATION_SCHEDULE
        )
        head_results[(layer, head)] = snapshots
        pbar.update(1)

pbar.close()
print(f"\n✓ Scanned {total_heads} heads.")

---
## 6. Visualization

### 6a. Head Resonance Grid
A heatmap of `12 × 12` showing how quickly each head converges. Bright = strong resonance.

In [ ]:
# ============================================================
# VIS 6a: HEAD RESONANCE GRID
# ============================================================

convergence_grid = np.zeros((model.cfg.n_layers, model.cfg.n_heads))

for (layer, head), snapshots in head_results.items():
    final_cos = snapshots[-1]["cosine_sim_to_prev"]
    convergence_grid[layer, head] = final_cos

fig_grid = px.imshow(
    convergence_grid,
    x=[f"H{h}" for h in range(model.cfg.n_heads)],
    y=[f"L{l}" for l in range(model.cfg.n_layers)],
    color_continuous_scale="Inferno",
    title="Head Resonance Grid: Final Cosine Similarity (iter 500)",
    labels={"color": "Cos Sim"},
    text_auto=".3f",
    aspect="auto",
)
fig_grid.update_layout(
    template="plotly_dark", height=600,
    xaxis_title="Head", yaxis_title="Layer",
)
fig_grid.show()

### 6b. Resonant Voice — What Token Does Each Head Sing?

In [ ]:
# ============================================================
# VIS 6b: RESONANT VOICE — Final Token Per Head (FIXED)
# ============================================================

md = "# Resonant Voice Map\n\n"
md += "Each cell shows the token that the head converges toward after 500 iterations.\n\n"
md += "| | " + " | ".join([f"**H{h}**" for h in range(model.cfg.n_heads)]) + " |\n"
md += "| :--- | " + " | ".join([":---:"] * model.cfg.n_heads) + " |\n"

for layer in range(model.cfg.n_layers):
    row = f"| **L{layer}** |"
    for head in range(model.cfg.n_heads):
        snapshots = head_results[(layer, head)]
        top_token = snapshots[-1]["top_tokens"][0][0]
        # Sanitize for Markdown
        clean = top_token.replace('\n', '\\n').replace('\r', '\\r').replace('`', "'").strip()
        if not clean:
            clean = '⎵'  # visible whitespace marker
        row += f" `{clean}` |"
    md += row + "\n"

display(Markdown(md))

### 6c. Selected Head Convergence Curves

In [ ]:
# ============================================================
# VIS 6c: SELECTED HEAD CONVERGENCE CURVES
# ============================================================

sorted_heads = sorted(
    head_results.keys(),
    key=lambda k: head_results[k][-1]["cosine_sim_to_prev"],
    reverse=True
)

interesting_heads = [
    sorted_heads[0],    # Fastest converging
    sorted_heads[-1],   # Slowest converging
    (9, 9),             # The EXP_002 'regeneration' head
    (0, 0),             # First head
    (5, 5),             # Mid-network
]
seen = set()
interesting_heads = [h for h in interesting_heads if not (h in seen or seen.add(h))]

fig_curves = go.Figure()
for (layer, head) in interesting_heads:
    snapshots = head_results[(layer, head)]
    iters = [s["iteration"] for s in snapshots]
    cos_sims = [s["cosine_sim_to_prev"] for s in snapshots]
    final_token = snapshots[-1]["top_tokens"][0][0].replace('\n', '\\n').strip()
    fig_curves.add_trace(go.Scatter(
        x=iters, y=cos_sims,
        mode='lines+markers',
        name=f"L{layer}.H{head} → '{final_token}'",
        marker=dict(size=8),
    ))

fig_curves.update_layout(
    title="Head Resonance: Convergence Curves (Selected Heads)",
    xaxis_title="Iteration",
    yaxis_title="Cosine Similarity to Previous",
    xaxis_type="log",
    template="plotly_dark",
    height=500,
)
fig_curves.add_hline(y=1.0, line_dash="dash", line_color="white", opacity=0.3)
fig_curves.show()

In [ ]:
# ============================================================
# STEP 7: SAVE ARTIFACTS
# ============================================================
import os

save_dir = os.path.join("..", "_DATA", "EXP_009")
os.makedirs(save_dir, exist_ok=True)

serializable = {}
for (layer, head), snapshots in head_results.items():
    key = f"L{layer}_H{head}"
    serializable[key] = {
        "iterations": [s["iteration"] for s in snapshots],
        "vectors": torch.stack([s["vector"] for s in snapshots]),
        "norms": [s["norm"] for s in snapshots],
        "cosine_sims": [s["cosine_sim_to_prev"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
    }

torch.save(serializable, os.path.join(save_dir, "009bFIX_head_loop_results.pt"))
torch.save(convergence_grid, os.path.join(save_dir, "009bFIX_convergence_grid.pt"))
print(f"[SAVED] {save_dir}/009bFIX_head_loop_results.pt")
print(f"[SAVED] {save_dir}/009bFIX_convergence_grid.pt")